# Homework 2: Vector Search

This notebook solves LLM Zoomcamp 2026 Homework 2 using the local ONNX embedder from the course materials. It uses the pinned course commit `8c1834d` and keeps the workflow self-contained inside `homework_02_vector_search/`.

## Setup And Data Loading

We load the lesson markdown files directly from the course repository with `GithubRepositoryDataReader`, and we reuse the same ONNX `Embedder` for every query and document embedding.

In [1]:
import time
from pathlib import Path

import numpy as np
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

from embedder import Embedder

In [2]:
COURSE_COMMIT = "8c1834d"
MODEL_DIR = Path("models/Xenova/all-MiniLM-L6-v2")

Q1_QUERY = "How does approximate nearest neighbor search work?"
Q4_QUERY = "What metric do we use to evaluate a search engine?"
Q5_QUERY = "How do I store vectors in PostgreSQL?"
Q6_QUERY = "How do I give the model access to tools?"
Q2_FILENAME = "02-vector-search/lessons/07-sqlitesearch-vector.md"

assert (MODEL_DIR / "tokenizer.json").exists(), "tokenizer.json is missing"
assert (MODEL_DIR / "model.onnx").exists(), "model.onnx is missing"
print("Model directory:", MODEL_DIR)

Model directory: models\Xenova\all-MiniLM-L6-v2


In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COURSE_COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = None
last_error = None
for attempt in range(1, 4):
    try:
        documents = [file.parse() for file in reader.read()]
        break
    except Exception as exc:
        last_error = exc
        print(f"Attempt {attempt} failed: {exc}")
        time.sleep(2)

if documents is None:
    raise last_error

print("Document count:", len(documents))
print("First three filenames:")
for filename in [doc["filename"] for doc in documents[:3]]:
    print(" -", filename)

Document count: 72
First three filenames:
 - 01-agentic-rag/lessons/01-intro.md
 - 01-agentic-rag/lessons/02-environment.md
 - 01-agentic-rag/lessons/03-rag.md


In [4]:
embedder = Embedder(MODEL_DIR)
embedder

## Q1: Embed A Query

We embed the query with the local ONNX model and inspect the vector shape, the first few values, and the L2 norm.

In [5]:
v = embedder.encode(Q1_QUERY)
q1_norm = float(np.linalg.norm(v))

print("Q1 query:", Q1_QUERY)
print("vector shape:", v.shape)
print("first five values:", v[:5].tolist())
print("v[0]:", float(v[0]))
print("vector L2 norm:", q1_norm)

Q1 query: How does approximate nearest neighbor search work?
vector shape: (384,)
first five values: [-0.02058203437252893, -0.0140458849235808, 0.030299406076994523, -0.05403784449888701, 0.07187811001608622]
v[0]: -0.02058203437252893
vector L2 norm: 1.0


## Q2: Cosine Similarity

Because the embedder returns normalized vectors by default, the dot product is also the cosine similarity.

In [6]:
target_doc = next(doc for doc in documents if doc["filename"] == Q2_FILENAME)
target_vector = embedder.encode(target_doc["content"])
q2_similarity = float(target_vector.dot(v))

print("filename:", target_doc["filename"])
print("similarity score:", q2_similarity)
print("nearest matching multiple-choice option:")
print("  The pinned course repository does not include the Week 2 option text, so use the exact score above to select the closest listed option on the homework page.")

filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
similarity score: 0.36107027225589694
nearest matching multiple-choice option:
  The pinned course repository does not include the Week 2 option text, so use the exact score above to select the closest listed option on the homework page.


## Chunking And Shared Search Helpers

For the remaining questions we chunk the lesson pages, embed chunk content in batches, and prepare both vector and keyword search indexes.

In [7]:
def preview(text, width=120):
    compact = " ".join(text.split())
    return compact[:width] + ("..." if len(compact) > width else "")


def print_ranked_results(results, score_lookup=None, limit=None):
    shown = results if limit is None else results[:limit]
    for rank, doc in enumerate(shown, start=1):
        key = (doc["filename"], doc.get("start"))
        score = score_lookup.get(key) if score_lookup else doc.get("score")
        print(f"{rank:>2}. filename={doc['filename']}")
        print(f"    start={doc.get('start')}")
        if score is not None:
            print(f"    score={float(score):.12f}")
        print(f"    preview={preview(doc['content'])}")


def vector_search_with_scores(vs, query_vector, num_results=5):
    results = vs.search(query_vector, num_results=num_results)
    return results


def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def rrf_score_map(result_lists, k=60):
    scores = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
    return scores

In [8]:
chunks = chunk_documents(documents, size=2000, step=1000)
print("Chunk count:", len(chunks))

batch_size = 32
chunk_batches = []
for start in range(0, len(chunks), batch_size):
    texts = [chunk["content"] for chunk in chunks[start:start + batch_size]]
    chunk_batches.append(embedder.encode_batch(texts))

X = np.vstack(chunk_batches)
print("Embedding matrix shape:", X.shape)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(vectors=X, payload=chunks)

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

Chunk count: 295


Embedding matrix shape: (295, 384)


## VectorSearch API Check

Before using `minsearch.VectorSearch`, we confirm the installed API shape so the notebook uses the correct constructor and method signatures.

In [9]:
import inspect

print("VectorSearch.__init__", inspect.signature(VectorSearch.__init__))
print("VectorSearch.fit", inspect.signature(VectorSearch.fit))
print("VectorSearch.search", inspect.signature(VectorSearch.search))

VectorSearch.__init__ (self, keyword_fields=None, numeric_fields=None, date_fields=None)
VectorSearch.fit (self, vectors, payload)
VectorSearch.search (self, query_vector, filter_dict=None, num_results=10, output_ids=False)


## Q3: Chunking And Manual Vector Search

We manually score each chunk with `scores = X.dot(v)` and inspect the top 10 highest-scoring chunks for the query from Q1.

In [10]:
q3_scores = X.dot(v)
q3_top_ids = np.argsort(q3_scores)[::-1][:10]
q3_results = [chunks[int(idx)] for idx in q3_top_ids]
q3_score_map = {
    (chunks[int(idx)]["filename"], chunks[int(idx)]["start"]): q3_scores[int(idx)]
    for idx in q3_top_ids
}

print_ranked_results(q3_results, score_lookup=q3_score_map)
print("Top result filename:", q3_results[0]["filename"])
print("Corresponding homework answer option:")
print("  If the options are filenames, select:", q3_results[0]["filename"])

 1. filename=02-vector-search/lessons/07-sqlitesearch-vector.md
    start=1000
    score=0.648901771858
    preview=rch. We score the query against every document and pick the top ones. It always finds the true top matches, but it pays ...
 2. filename=01-agentic-rag/lessons/05-search.md
    start=0
    score=0.551034634844
    preview=# Search Video: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)...
 3. filename=04-evaluation/lessons/05-search-metrics.md
    start=1000
    score=0.406560707206
    preview=k mean the same thing. Let's calculate it: ```python cnt = 0 for line in example: if 1 in line: cnt = cnt + 1 cnt ``` Th...
 4. filename=02-vector-search/lessons/04-vector-search.md
    start=0
    score=0.406181979024
    preview=# Vector Search Video: [Watch this lesson](https://www.youtube.com/watch?v=h-_tdBc24qc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8...
 5. filename=06-best-practices/lessons/03-reranking.md
    start=0
    scor

## Q4: Vector Search With minsearch

Now we use `minsearch.VectorSearch` on the same chunk embeddings and inspect the top 5 results for a new query.

In [11]:
q4_vector = embedder.encode(Q4_QUERY)
q4_results = vector_index.search(q4_vector, num_results=5)
print_ranked_results(q4_results)
print("Top result filename:", q4_results[0]["filename"])
print("Corresponding homework answer option:")
print("  If the options are filenames, select:", q4_results[0]["filename"])

 1. filename=04-evaluation/lessons/05-search-metrics.md
    start=0
    preview=# Search Evaluation Metrics Video: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFN...
 2. filename=04-evaluation/lessons/01-intro.md
    start=0
    preview=# Evaluation Video: [Watch this lesson](https://www.youtube.com/watch?v=eC_IcxfxoiQ&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0...
 3. filename=01-agentic-rag/lessons/05-search.md
    start=0
    preview=# Search Video: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)...
 4. filename=04-evaluation/lessons/01-intro.md
    start=2000
    preview=Offline evaluation: run the system on a test dataset and compute metrics - Online evaluation: collect feedback from real...
 5. filename=04-evaluation/lessons/15-next-steps.md
    start=0
    preview=# Next Steps Video: [Watch this lesson](https://www.youtube.com/watch?v=TlKPBjItUw8&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0...
Top re

## Q5: Text Search Versus Vector Search

We compare the top 5 chunk results from vector search and keyword search for a PostgreSQL vector-storage question.

In [12]:
q5_vector = vector_index.search(embedder.encode(Q5_QUERY), num_results=5)
q5_text = text_index.search(Q5_QUERY, num_results=5)

q5_vector_files = [doc["filename"] for doc in q5_vector]
q5_text_files = [doc["filename"] for doc in q5_text]
q5_vector_only = sorted(set(q5_vector_files) - set(q5_text_files))
q5_text_only = sorted(set(q5_text_files) - set(q5_vector_files))

print("vector-search filenames:")
for filename in q5_vector_files:
    print(" -", filename)

print("\ntext-search filenames:")
for filename in q5_text_files:
    print(" -", filename)

print("\nfilenames found only in vector results:", q5_vector_only)
print("filenames found only in text results:", q5_text_only)
print("Answer option present in vector results but absent from text-search results:")
print(" ", q5_vector_only[0] if q5_vector_only else "None")

vector-search filenames:
 - 02-vector-search/lessons/08-pgvector.md
 - 02-vector-search/lessons/08-pgvector.md
 - 03-orchestration/lessons/05-rag.md
 - 02-vector-search/lessons/08-pgvector.md
 - 02-vector-search/lessons/08-pgvector.md

text-search filenames:
 - 02-vector-search/lessons/02-embeddings.md
 - 03-orchestration/lessons/05-rag.md
 - 02-vector-search/lessons/01-intro.md
 - 03-orchestration/lessons/05-rag.md
 - 02-vector-search/lessons/01-intro.md

filenames found only in vector results: ['02-vector-search/lessons/08-pgvector.md']
filenames found only in text results: ['02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/02-embeddings.md']
Answer option present in vector results but absent from text-search results:
  02-vector-search/lessons/08-pgvector.md


## Q6: Hybrid Search With Reciprocal Rank Fusion

We retrieve vector and keyword results for the tools-access question, fuse them with the exact RRF function supplied in the homework, and inspect the top fused results.

In [13]:
q6_vector_results = vector_index.search(embedder.encode(Q6_QUERY), num_results=10)
q6_text_results = text_index.search(Q6_QUERY, num_results=10)
q6_rrf_results = rrf([q6_vector_results, q6_text_results], k=60, num_results=10)
q6_rrf_scores = rrf_score_map([q6_vector_results, q6_text_results], k=60)

print_ranked_results(q6_rrf_results, score_lookup=q6_rrf_scores)
print("Top fused filename:", q6_rrf_results[0]["filename"])
print("Corresponding homework answer option:")
print("  If the options are filenames, select:", q6_rrf_results[0]["filename"])

 1. filename=01-agentic-rag/lessons/13-function-calling.md
    start=4000
    score=0.032018442623
    preview=function. `parameters` is a JSON schema for the arguments, and we mark `query` as required so the model always fills it ...
 2. filename=01-agentic-rag/lessons/13-function-calling.md
    start=2000
    score=0.030090497738
    preview=see what the LLM does without any tools. We ask it a course-specific question and look at the answer. ```python messages...
 3. filename=01-agentic-rag/lessons/01-intro.md
    start=2000
    score=0.016666666667
    preview=wrong. ## The project RAG solves these problems by giving the LLM relevant documents at question time. We don't hope the...
 4. filename=01-agentic-rag/lessons/14-agentic-loop.md
    start=0
    score=0.016666666667
    preview=# The Agentic Loop Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDB...
 5. filename=04-evaluation/lessons/02-ground-truth.md
    start=1000
    score

## Computed Answers Summary

This final cell collects the exact computed values that are needed for the homework submission.

In [14]:
summary = {
    "Q1_v0": float(v[0]),
    "Q2_similarity": q2_similarity,
    "Q3_top_filename": q3_results[0]["filename"],
    "Q4_top_filename": q4_results[0]["filename"],
    "Q5_vector_only": q5_vector_only,
    "Q6_top_filename": q6_rrf_results[0]["filename"],
}
summary

{'Q1_v0': -0.02058203437252893,
 'Q2_similarity': 0.36107027225589694,
 'Q3_top_filename': '02-vector-search/lessons/07-sqlitesearch-vector.md',
 'Q4_top_filename': '04-evaluation/lessons/05-search-metrics.md',
 'Q5_vector_only': ['02-vector-search/lessons/08-pgvector.md'],
 'Q6_top_filename': '01-agentic-rag/lessons/13-function-calling.md'}